# clean
> Strip superfluous metadata from notebooks
- order: 11

In [ ]:
#| default_exp clean

In [ ]:
#| export
import ast,warnings,stat
from astunparse import unparse
from textwrap import indent

from fastcore.nbio import *
from fastcore.nbio import _directive, _dir_line, _meta_directives, _unparse_dir
from fastcore.script import *
from fastcore.utils import *
from fastcore.xtras import *

from nbdev.imports import *
from nbdev.config import *
from nbdev.sync import *
from nbdev.process import first_code_ln

In [ ]:
#| hide
from fastcore.test import *

To avoid pointless conflicts while working with jupyter notebooks (with different execution counts or cell metadata), it is recommended to clean the notebooks before committing anything (done automatically if you install the git hooks with `nbdev-install-hooks`). The following functions are used to do that. Cleaning also adds cell `id`s if missing (required by nbformat 4.5+).

## Trust

In [ ]:
#| export
@call_parse
def nbdev_trust(
    fname:str=None,  # A notebook name or glob to trust
    force_all:bool=False  # Also trust notebooks that haven't changed
):
    "Trust notebooks matching `fname`."
    try: from nbformat.sign import NotebookNotary
    except:
        import warnings
        warnings.warn("Please install jupyter and try again")
        return
    from nbformat import read

    fname = Path(fname if fname else get_config().nbs_path)
    path = fname if fname.is_dir() else fname.parent
    check_fname = path/".last_checked"
    last_checked = os.path.getmtime(check_fname) if check_fname.exists() else None
    nbs = globtastic(fname, file_glob='*.ipynb', skip_folder_re='^[_.]') if fname.is_dir() else [fname]
    for fn in nbs:
        if last_checked and not force_all:
            last_changed = os.path.getmtime(fn)
            if last_changed < last_checked: continue
        with open(fn, 'r', encoding='utf-8') as f: nb = read(f, as_version=4)
        if not NotebookNotary().check_signature(nb): NotebookNotary().sign(nb)
    check_fname.touch(exist_ok=True)

## Clean

### Utils -

In [ ]:
#| export
_repr_id_re = re.compile('(<.*?)( at 0x[0-9a-fA-F]+)(>)')

_sub = partial(_repr_id_re.sub, r'\1\3')

def _skip_or_sub(x): return _sub(x) if "at 0x" in x else x

def _clean_cell_output_id(lines):
    return _skip_or_sub(lines) if isinstance(lines,str) else [_skip_or_sub(o) for o in lines]

In [ ]:
#| hide
test_eq(_clean_cell_output_id(['Lambda(func=<function _add2 at 0x7f8252378820>)',
                               '[<PIL.Image.Image image mode=RGB size=320x240 at 0x7FAC4E2CF610>,\n',
                               '(<a at 0x7f8252378820>, <b at 0x7EFE94247550>, <c at 0x7f8252378820>)']),
                              ['Lambda(func=<function _add2>)',
                               '[<PIL.Image.Image image mode=RGB size=320x240>,\n',
                               '(<a>, <b>, <c>)'])
test_eq(_clean_cell_output_id('foo\n<function _add2 at 0x7f8252378820>\nbar'), 'foo\n<function _add2>\nbar')

In [ ]:
#| export
def _clean_cell_output(cell, clean_ids, allowed_out_meta_keys):
    "Remove `cell` output execution count and optionally ids from text reprs"
    outputs = cell.get('outputs', [])
    for o in outputs:
        if 'execution_count' in o: o['execution_count'] = None
        data = o.get('data', {})
        data.pop("application/vnd.google.colaboratory.intrinsic+json", None)
        for k in data:
            if k.startswith('text') and clean_ids: data[k] = _clean_cell_output_id(data[k])
            if k.startswith('image') and "svg" not in k: data[k] = data[k].rstrip()
        if 'text' in o and clean_ids: o['text'] = _clean_cell_output_id(o['text'])
        if 'metadata' in o: o['metadata'] = {k:v for k,v in o['metadata'].items() if k in allowed_out_meta_keys}

In [ ]:
#| export
def _clean_cell(cell, clear_all, allowed_metadata_keys, clean_ids, allowed_out_meta_keys):
    "Clean `cell` by removing superfluous metadata or everything except the input if `clear_all`"
    if 'execution_count' in cell: cell['execution_count'] = None
    if 'outputs' in cell:
        if clear_all: cell['outputs'] = []
        else:         _clean_cell_output(cell, clean_ids, allowed_out_meta_keys)
    if cell['source'] == ['']: cell['source'] = []
    cell['metadata'] = {} if clear_all else {
        k:v for k,v in cell['metadata'].items() if k in allowed_metadata_keys}
    if 'id' not in cell: cell['id'] = rtoken_hex(4)

In [ ]:
#| export
def clean_nb(
    nb, # The notebook to clean
    clear_all=False, # Remove all cell metadata and cell outputs?
    allowed_metadata_keys:list=None, # Preserve the list of keys in the main notebook metadata
    allowed_cell_metadata_keys:list=None, # Preserve the list of keys in cell level metadata
    clean_ids=True, # Remove ids from plaintext reprs?
    allowed_out_metadata_keys:list=None, # Preserve the list of keys in output metadata
    repair:bool=True, # Fix structural problems first (see `repair_nb`)?
):
    "Clean `nb` from superfluous metadata"
    if repair: repair_nb(nb)
    metadata_keys = {"kernelspec", "jekyll", "jupytext", "doc", "widgets", "nbdev"}
    if allowed_metadata_keys: metadata_keys.update(allowed_metadata_keys)
    cell_metadata_keys = {"hide_input", "nbdev"}
    if allowed_cell_metadata_keys: cell_metadata_keys.update(allowed_cell_metadata_keys)
    out_meta_keys = set()
    if allowed_out_metadata_keys: out_meta_keys.update(allowed_out_metadata_keys)
    for c in nb['cells']: _clean_cell(c, clear_all, cell_metadata_keys, clean_ids, out_meta_keys)
    if nb.get('metadata', {}).get('kernelspec', {}).get('name', None):
        nb['metadata']['kernelspec']['display_name'] = nb["metadata"]["kernelspec"]["name"]
    nb['metadata'] = {k:v for k,v in nb['metadata'].items() if k in metadata_keys}
    # Cell IDs were added in nbformat 4.5
    if nb.get('nbformat') == 4 and nb.get('nbformat_minor', 0) < 5: nb['nbformat_minor'] = 5

Jupyter adds a trailing <code>\n</code> to images in cell outputs. Vscode-jupyter does not.\
Notebooks should be brought to a common style to avoid unnecessary diffs:

In [ ]:
test_nb = read_nb('../../tests/image.ipynb')
assert test_nb.cells[0].outputs[0].data['image/png'][-1] == "\n" # Make sure it was not converted by acccident
clean_nb(test_nb)
assert test_nb.cells[0].outputs[0].data['image/png'][-1] != "\n"

The test notebook has metadata in both the main metadata section and contains cell level metadata in the second cell:

In [ ]:
test_nb = read_nb('../../tests/metadata.ipynb')

assert {'meta', 'jekyll', 'nbdev', 'my_extra_key', 'my_removed_key'} <= test_nb.metadata.keys()
assert {'meta', 'hide_input', 'my_extra_cell_key', 'nbdev', 'my_removed_cell_key'} == test_nb.cells[1].metadata.keys()

After cleaning the notebook, all extra metadata is removed, only some keys are allowed by default:

In [ ]:
clean_nb(test_nb)

assert {'jekyll', 'kernelspec', 'nbdev'} == test_nb.metadata.keys()
assert {'hide_input', 'nbdev'} == test_nb.cells[1].metadata.keys()

`clean_nb` also repairs structural problems by default (via `repair_nb`), so notebooks that Jupyter would reject, such as a markdown cell carrying an `outputs` attr, are fixed on every clean:

In [ ]:
_nb = dict2nb(dict(cells=[dict(cell_type='markdown', source='hi', outputs=[], execution_count=1, id='m1', metadata={})],
                   metadata=dict(kernelspec=dict(name='python3', display_name='Python 3')), nbformat=4, nbformat_minor=5))
clean_nb(_nb)
assert 'outputs' not in _nb.cells[0] and 'execution_count' not in _nb.cells[0]
validate_nb(_nb)

We can preserve some additional keys at the notebook or cell levels:

In [ ]:
test_nb = read_nb('../../tests/metadata.ipynb')
clean_nb(test_nb, allowed_metadata_keys={'my_extra_key'}, allowed_cell_metadata_keys={'my_extra_cell_key'})

assert {'jekyll', 'kernelspec', 'nbdev', 'my_extra_key'} == test_nb.metadata.keys()
assert {'hide_input', 'nbdev', 'my_extra_cell_key'} == test_nb.cells[1].metadata.keys()

Passing `clear_all=True` removes everything from the cell metadata:

In [ ]:
test_nb = read_nb('../../tests/metadata.ipynb')
clean_nb(test_nb, clear_all=True)

assert {'jekyll', 'kernelspec', 'nbdev'} == test_nb.metadata.keys()
test_eq(test_nb.cells[1].metadata, {})

Passing `clean_ids=True` removes `id`s from plaintext repr outputs, to avoid notebooks whose contents change on each run since they often lead to git merge conflicts. For example:

```
<PIL.PngImagePlugin.PngImageFile image mode=L size=28x28 at 0x7FB4F8979690>
```

becomes:

```
<PIL.PngImagePlugin.PngImageFile image mode=L size=28x28>
```

*Cell* IDs, on the other hand, are always added if missing

In [ ]:
test_cell = {'source': 'x=1', 'cell_type': 'code', 'metadata': {}}
_clean_cell(test_cell, False, set(), True, set())
test_cell['id']

'64e20756'

### Commands -

In [ ]:
#| export
def _reconfigure(*strms):
    for s in strms:
        if hasattr(s,'reconfigure'): s.reconfigure(encoding='utf-8')

In [ ]:
#| export
def process_write(warn_msg, proc_nb, f_in, f_out=None, disp=False):
    if not f_out: f_out = f_in
    if isinstance(f_in, (str,Path)): f_in = Path(f_in).open(encoding="utf-8")
    try:
        _reconfigure(f_in, f_out)
        nb = loads(f_in.read())
        proc_nb(nb)
        write_nb(nb, f_out) if not disp else sys.stdout.write(nb2str(nb))
    except Exception as e:
        warn(f'{warn_msg}')
        warn(e)

## Directive migrations

Deliberate, opt-in rewrites for moving to the directive conventions introduced in v3.3: directives can live in cell metadata (an `nbdev` dict key), notebook-scope directives like `default_exp` can live in notebook metadata, and comment spellings have one canonical form. None of these run by default; they're flags on `nbdev-clean` for one-off migration runs, so the git hook never triggers them.

In [ ]:
#| export
def _cmt_dirs(cell):
    "Comment directives in `cell` as `{name: value}`, plus the partitioned `(dirs,code)` lines"
    dirs,code = cell._partition()
    return dict(t for t in (_directive(s, cell.lang_) for s in dirs) if t),dirs,code

def _rm_dir_lines(cell, dirs, code, names):
    "Rewrite `cell` source without the directive lines in `names`"
    cell.set_source(''.join([o for o in dirs if (t:=_directive(o, cell.lang_)) is None or t[0] not in names] + code))

def _to_meta(cell, names):
    "Move comment directives in `names` to the cell's `nbdev` metadata key"
    cmts,dirs,code = _cmt_dirs(cell)
    move = {k:v for k,v in cmts.items() if k in names}
    if not move: return
    cell.setdefault('metadata',{}).setdefault('nbdev',{}).update({k:_unparse_dir(v) for k,v in move.items()})
    _rm_dir_lines(cell, dirs, code, move)

def _to_comments(cell, names):
    "Move directives in `names` from the cell's `nbdev` metadata key to comments"
    move = {k:v for k,v in _meta_directives(cell).items() if k in names}
    if not move: return
    nbd = cell.metadata['nbdev']
    for k in move: nbd.pop(k, None)
    if not nbd: del cell.metadata['nbdev']
    dirs,code = cell._partition()
    cell.set_source(''.join(dirs + [_dir_line(k, v, cell.lang_) for k,v in move.items()] + code))

`_to_meta` and `_to_comments` move named directives between a cell's comments and its `nbdev` metadata key, preserving values through the same mapping the parser uses (bare directives become `true`, and so on):

In [ ]:
c = mk_cell('#| hide\n#| eval: false\n#| export: utils\n1+1')
_to_meta(c, ['hide','eval'])
test_eq(c.metadata['nbdev'], dict(hide='true', eval='false'))
test_eq(c.source, '#| export: utils\n1+1')

_to_comments(c, ['hide','eval'])
assert 'nbdev' not in c.metadata
test_eq(c.directives, {'export':'utils', 'hide':'', 'eval':'false'})

In [ ]:
#| export
def _canon_dirs(cell):
    "Rewrite `cell`'s comment directives in canonical form (colon-separated, bare for true)"
    dirs,code = cell._partition()
    new = [o if (t:=_directive(o, cell.lang_)) is None else _dir_line(*t, lang=cell.lang_) for o in dirs]
    if new != dirs: cell.set_source(''.join(new + code))

def _hoist_nb_meta(nb, names=('default_exp',)):
    "Move notebook-scope directives in `names` to notebook metadata, dropping any cell left empty"
    for c in list(nb.cells):
        cmts,dirs,code = _cmt_dirs(c)
        move = {k:v for k,v in cmts.items() if k in names}
        if not move: continue
        nb.setdefault('metadata',{}).setdefault('nbdev',{}).update({k:_unparse_dir(v) for k,v in move.items()})
        _rm_dir_lines(c, dirs, code, move)
        left = set(c.directives) - {'hide'}
        if not left and not ''.join(c._partition()[1]).strip(): nb.cells.remove(c)

def _dir_moves(nb, dirs=False, to_meta=None, to_comments=None, nb_meta=False):
    "Apply directive migrations to loaded notebook dict `nb` in place"
    nbo = dict2nb(nb)
    if to_meta:
        for c in nbo.cells: _to_meta(c, to_meta.split())
    if to_comments:
        for c in nbo.cells: _to_comments(c, to_comments.split())
    if nb_meta: _hoist_nb_meta(nbo)
    if dirs:
        for c in nbo.cells: _canon_dirs(c)
    nb['cells'],nb['metadata'] = nbo.cells,nbo.metadata

`_canon_dirs` respells each directive line canonically without touching anything else, and `_hoist_nb_meta` moves `default_exp` to notebook metadata, deleting a first cell that held nothing but directives (a bare `#| hide` on an otherwise-empty cell hides nothing). `_dir_moves` bundles the migrations for the CLI, converting the raw loaded dict to `NbCell`s first:

In [ ]:
c = mk_cell('#| default_exp core\n#| eval:false\n1+1')
_canon_dirs(c)
test_eq(c.source, '#| default_exp: core\n#| eval: false\n1+1')

_nb = dict2nb(dict(cells=[mk_cell('#| hide\n#| default_exp: core'), mk_cell('#| export\n1+1')],
                   metadata={}, nbformat=4, nbformat_minor=5))
_hoist_nb_meta(_nb)
test_eq(_nb.metadata['nbdev'], dict(default_exp='core'))
test_eq(len(_nb.cells), 1)
test_eq(_nb.cells[0].source, '#| export\n1+1')

In [ ]:
#| export
def _nbdev_clean(nb, path=None, clear_all=None, repair=True, dirs=False, to_meta=None, to_comments=None, nb_meta=False):
    cfg = get_config(path=path)
    clear_all = clear_all or cfg.clear_all
    allowed_metadata_keys = cfg.get("allowed_metadata_keys") or []
    allowed_cell_metadata_keys = cfg.get("allowed_cell_metadata_keys") or []
    allowed_out_metadata_keys = cfg.get("allowed_out_metadata_keys") or []
    if dirs or to_meta or to_comments or nb_meta: _dir_moves(nb, dirs, to_meta, to_comments, nb_meta)
    clean_nb(nb, clear_all, allowed_metadata_keys, allowed_cell_metadata_keys, cfg.clean_ids, allowed_out_metadata_keys, repair=repair)
    if path: nbdev_trust.__wrapped__(path)

In [ ]:
#| export
@call_parse
def nbdev_clean(
    fname:str=None, # A notebook name or glob to clean
    clear_all:bool=False, # Remove all cell metadata and cell outputs?
    disp:bool=False,  # Print the cleaned outputs
    stdin:bool=False, # Read notebook from input stream
    repair:bool_arg=True, # Fix structural problems, e.g. stray outputs on non-code cells (see `repair_nb`)?
    dirs:bool=False, # Rewrite comment directives in canonical form?
    to_meta:str=None, # Space-separated directive names to move from comments to cell metadata
    to_comments:str=None, # Space-separated directive names to move from cell metadata to comments
    nb_meta:bool=False # Move `default_exp` into notebook metadata?
):
    "Clean all notebooks in `fname` to avoid merge conflicts"
    # Git hooks will pass the notebooks in stdin
    _clean = partial(_nbdev_clean, clear_all=clear_all, repair=repair, dirs=dirs, to_meta=to_meta, to_comments=to_comments, nb_meta=nb_meta)
    _write = partial(process_write, warn_msg='Failed to clean notebook', proc_nb=_clean)
    if stdin: return _write(f_in=sys.stdin, f_out=sys.stdout)
    if fname is None: fname = get_config().nbs_path
    for f in globtastic(fname, file_glob='*.ipynb', skip_folder_re='^[_.]'): _write(f_in=f, disp=disp)

By default (`fname` left to `None`), all the notebooks in `config.nbs_path` are cleaned. You can opt in to fully clean the notebook by removing every bit of metadata and the cell outputs by passing `clear_all=True`.

If you want to keep some keys in the main notebook metadata you can set `allowed_metadata_keys` in `[tool.nbdev]` in `pyproject.toml`.
Similarly for cell level metadata use `allowed_cell_metadata_keys`, and for output metadata use `allowed_out_metadata_keys`. For example, to preserve both `k1` and `k2` at both the notebook and cell level add the following to `pyproject.toml`:
```toml
[tool.nbdev]
allowed_metadata_keys = ["k1", "k2"]
allowed_cell_metadata_keys = ["k1", "k2"]
allowed_out_metadata_keys = ["k1", "k2"]
```

### Jupyter -

In [ ]:
#| export
def clean_jupyter(path, model, **kwargs):
    "Clean Jupyter `model` pre save to `path`"
    if not (model['type']=='notebook' and model['content']['nbformat']==4): return
    jupyter_hooks = get_config(path=path).jupyter_hooks
    if jupyter_hooks: _nbdev_clean(model['content'], path=path)

This cleans notebooks on-save to avoid unnecessary merge conflicts. The easiest way to install it for both Jupyter Notebook and Lab is by running  `nbdev-install-hooks`. It works by implementing a `pre_save_hook` from Jupyter's [file save hook API](https://jupyter-server.readthedocs.io/en/latest/developers/savehooks.html).

## Hooks

In [ ]:
#| export
_pre_save_hook_src = '''
def nbdev_clean_jupyter(**kwargs):
    try: from nbdev.clean import clean_jupyter
    except ModuleNotFoundError: return
    clean_jupyter(**kwargs)

c.ContentsManager.pre_save_hook = nbdev_clean_jupyter'''.strip()
_pre_save_hook_re = re.compile(r'c\.(File)?ContentsManager\.pre_save_hook')

In [ ]:
#| export
def _add_jupyter_hooks(src, path):
    if _pre_save_hook_src in src: return
    mod = ast.parse(src)
    for node in ast.walk(mod):
        if not isinstance(node,ast.Assign): continue
        target = only(node.targets)
        if _pre_save_hook_re.match(unparse(target)):
            pre = ' '*2
            old = indent(unparse(node), pre)
            new = indent(_pre_save_hook_src, pre)
            sys.stderr.write(f"Can't install hook to '{path}' since it already contains:\n{old}\n"
                             f"Manually update to the following (without indentation) for this functionality:\n\n{new}\n\n")
            return
    src = src.rstrip()
    if src: src+='\n\n'
    return src+_pre_save_hook_src

In [ ]:
#| hide
# Returns None if hook is already installed
res = _add_jupyter_hooks(_pre_save_hook_src, 'config.py')
test_is(res, None)

In [ ]:
#| hide
# Returns None and warns if pre_save_hook is already set
res = _add_jupyter_hooks("c.ContentsManager.pre_save_hook = my_hook\n", 'config.py')
test_is(res, None)

Can't install hook to 'config.py' since it already contains:

  c.ContentsManager.pre_save_hook = my_hook

Manually update to the following (without indentation) for this functionality:

  def nbdev_clean_jupyter(**kwargs):
      try: from nbdev.clean import clean_jupyter
      except ModuleNotFoundError: return
      clean_jupyter(**kwargs)

  c.ContentsManager.pre_save_hook = nbdev_clean_jupyter



In [ ]:
#| hide
# Adds after existing source
show_src(_add_jupyter_hooks('an_existing_line = True\n', 'config.py'))

<div class="prose" markdown="1">

```python
an_existing_line = True

def nbdev_clean_jupyter(**kwargs):
    try: from nbdev.clean import clean_jupyter
    except ModuleNotFoundError: return
    clean_jupyter(**kwargs)

c.ContentsManager.pre_save_hook = nbdev_clean_jupyter
```

</div>

In [ ]:
#| export
def _git_root(): 
    try: return Path(run('git rev-parse --show-toplevel'))
    except OSError: return None

In [ ]:
#| hide
import tempfile

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as d, working_directory(d): test_is(_git_root(), None)

In [ ]:
#| export
def _add_attrs(path, attrs):
    "Append missing attribute lines to git attributes file at `path`"
    txt = path.read_text() if path.exists() else ''
    have = [l.split() for l in txt.splitlines()]  # whitespace-insensitive: nbdime writes its lines with tabs
    for attr in attrs:
        if attr.split() not in have:
            if txt and not txt.endswith('\n'): txt+='\n'
            txt += attr+'\n'
    path.write_text(txt)

def _cfg_drivers(loc, merge, diff):
    "Define the `jupyternotebook` merge/diff drivers via `git config <loc>`"
    if merge:
        run(f'git config {loc} merge.jupyternotebook.name "resolve conflicts with nbdev_fix"')
        run(f'git config {loc} merge.jupyternotebook.driver "nbdev-merge %O %A %B %P"')
    if diff: run(f'git config {loc} diff.jupyternotebook.command nbdev-diff-driver')

@call_parse
def nbdev_install_hooks(
    merge:bool_arg=True, # Install the notebook merge driver?
    diff:bool_arg=True, # Install the notebook diff driver?
    globally:bool_arg=False # Define the drivers in `~/.gitconfig` and the global attributes file, instead of repo files?
):
    "Install Jupyter and git hooks to automatically clean, trust, and fix merge conflicts in notebooks"
    cfg_path = Path.home()/'.jupyter'
    cfg_path.mkdir(exist_ok=True)
    cfg_fns = [cfg_path/f'jupyter_{o}_config.py' for o in ('notebook','server')]
    for fn in cfg_fns:
        src = fn.read_text() if fn.exists() else ''
        upd = _add_jupyter_hooks(src, fn)
        if upd is not None: fn.write_text(upd)

    nbdev_attrs = (['*.ipynb merge=jupyternotebook'] if merge else []) + (['*.ipynb diff=jupyternotebook'] if diff else [])
    if globally:
        _cfg_drivers('--global', merge, diff)
        rc,p = run('git config --global --get core.attributesFile', ignore_ex=True)
        if rc: p = os.environ.get('XDG_CONFIG_HOME', '~/.config') + '/git/attributes'
        attrs_path = Path(p).expanduser()
        attrs_path.parent.mkdir(parents=True, exist_ok=True)
        _add_attrs(attrs_path, nbdev_attrs)
        return print("Hooks are installed globally.")

    repo_path = _git_root()
    if repo_path is None:
        sys.stderr.write('Not in a git repository, git hooks cannot be installed.\n')
        return
    hook_path = repo_path/'.git'/'hooks'
    fn = hook_path/'post-merge'
    hook_path.mkdir(parents=True, exist_ok=True)
    fn.write_text("#!/bin/bash\nnbdev-trust")
    os.chmod(fn, os.stat(fn).st_mode | stat.S_IEXEC)

    cmd = 'git config --local include.path ../.gitconfig'
    cfg_fn = repo_path/'.gitconfig'
    cfg_fn.write_text(f'''# Generated by nbdev-install-hooks
#
# If you need to disable this instrumentation do:
#   git config --local --unset include.path
#
# To restore:
#   {cmd}
#
''')
    _cfg_drivers(f'--file "{cfg_fn}"', merge, diff)
    run(cmd)

    _add_attrs(repo_path/'.gitattributes', nbdev_attrs)

    print("Hooks are installed.")

See `clean_jupyter` and `nbdev-merge` for more about how each hook works.

Both git drivers are registered under the name `jupyternotebook`, the same name nbdime uses. Repo installs define them in a repo-local `.gitconfig` (wired in via `include.path`) and activate them in the committed `.gitattributes`; `--globally` instead defines them in `~/.gitconfig` and activates them in the global attributes file (`core.attributesFile`, defaulting to `~/.config/git/attributes`). Since the attribute lines just name a driver, a committed `.gitattributes` means "use your preferred notebook driver": whichever tool defined `jupyternotebook` last in a config git consults wins, so switching between nbdev and nbdime is just re-running either tool's enable command. Global installs skip the repo-only `post-merge` trust hook.

In [ ]:
#| hide
with tempfile.TemporaryDirectory() as d, working_directory(d):
    run('git init -q')
    nbdev_install_hooks(diff=False)
    cfgtxt = (Path(d)/'.gitconfig').read_text()
    assert '[merge "jupyternotebook"]' in cfgtxt and 'diff "jupyternotebook"' not in cfgtxt
    nbdev_install_hooks(merge=False)
    cfgtxt = (Path(d)/'.gitconfig').read_text()
    assert '[diff "jupyternotebook"]' in cfgtxt and 'merge "jupyternotebook"' not in cfgtxt
    nbdev_install_hooks()
    cfgtxt = (Path(d)/'.gitconfig').read_text()
    assert '[merge "jupyternotebook"]' in cfgtxt and '[diff "jupyternotebook"]' in cfgtxt
    attrs = (Path(d)/'.gitattributes').read_text()
    assert '*.ipynb merge=jupyternotebook\n' in attrs and '*.ipynb diff=jupyternotebook\n' in attrs

    os.environ['GIT_CONFIG_GLOBAL'],os.environ['XDG_CONFIG_HOME'] = str(Path(d)/'gcfg'),str(Path(d)/'xdg')
    try:
        nbdev_install_hooks(globally=True)
        assert 'jupyternotebook' in (Path(d)/'gcfg').read_text()
        attrs = (Path(d)/'xdg'/'git'/'attributes').read_text()
        assert '*.ipynb merge=jupyternotebook\n' in attrs and '*.ipynb diff=jupyternotebook\n' in attrs
    finally: del os.environ['GIT_CONFIG_GLOBAL'], os.environ['XDG_CONFIG_HOME']

## End-to-end git hooks test -

In [ ]:
#| hide
def _git_brunch_current(): return run('git branch --show-current')

In [ ]:
#| hide
meta = {'nbformat': 4,'metadata':{'kernelspec':{'display_name':'Python 3','language': 'python','name': 'python3'}}}
base = dict2nb({'cells':[mk_cell('import random'),
                         mk_cell('random.random()')], **meta})
base.cells[-1].output = create_output('0.3314001088639852\n0.20280244713400464', 'plain')

In [ ]:
#| hide
from copy import deepcopy

In [ ]:
#| hide
ours = deepcopy(base)
ours.cells[0].source+=',os' # Change first cell
ours.cells.insert(1, mk_cell('Calculate a random number:', cell_type='markdown')) # New cell
ours.cells[-1].output = create_output('0.3379097372590093\n0.7379492349993123', 'plain') # Change outputs

In [ ]:
#| hide
thrs = deepcopy(base)
thrs.cells[0].source+=',sys'# Also change first cell
thrs.cells.insert(0, mk_cell('# Random numbers', cell_type='markdown')) # New cell
thrs.cells[-1].output = create_output('0.6587181429602441\n0.5962200692415515', 'plain') # Change outputs

In [ ]:
#| hide
import subprocess

In [ ]:
#| hide
def _run(cmd, check=True):
    proc = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if check and proc.returncode != 0:
        msg = f"Command '{cmd}' returned non-zero exit status {proc.returncode}"
        if proc.stdout.strip(): msg+=f'\nstdout: {proc.stdout.strip()}'
        if proc.stderr.strip(): msg+=f'\nstderr: {proc.stderr.strip()}'
        raise RuntimeError(msg)
    return proc

In [ ]:
#| hide
#| eval: false
with tempfile.TemporaryDirectory() as d, working_directory(d):
    _run('git init')
    _run("git config user.email 'nbdev@fast.ai'")
    _run("git config user.name 'nbdev'")

    nbs_path = Path('nbs')
    nbs_path.mkdir()
    Path('pyproject.toml').write_text('[tool.nbdev]\\nnbs_path = "nbs"')
    _run('nbdev-install-hooks')
    
    fn = 'random.ipynb'
    p = nbs_path/fn
    write_nb(base, p)
    _run(f"git add . && git commit -m 'add {fn}'")
    default = _git_brunch_current()

    feature = 'add-heading'
    _run(f'git checkout -b {feature}')
    write_nb(thrs, p)
    _run("git commit -am 'heading'")

    _run(f'git checkout {default}')
    write_nb(ours, p)
    _run("git commit -am 'docs'")

    proc = _run(f'git merge {feature}', check=False)
    if proc.stderr: raise AssertionError(f'Git hook failed with:\n\n{proc.stderr}')
    assert proc.returncode != 0, proc.stdout.strip() # Should error since we can't autofix cell source change
    nb = read_nb(p)

s = [o.source for o in nb.cells]
test_eq(s, ['# Random numbers',
            '`<<<<<<< HEAD`',
           'import random,os',
           'Calculate a random number:',
           '`=======`',
           'import random,sys',
           '`>>>>>>> add-heading`',
           'random.random()'])
test_eq(nb.cells[-1].output, ours.cells[-1].output)

## Export -

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()